# Extraer base de datos de estudiantes matriculados


In [1]:
import pandas as pd
# 1. Cargar la base
df = pd.read_csv("basematricula2019_2021.csv")

# 2. Filtrar los periodos que te interesan
periodos_validos = ['2020-2', '2021-1', '2021-2']
df_filtrado = df[df['periodo_matricula'].isin(periodos_validos)]

# 3. Limpiar la columna DNI:
# - Asegurarte que sea string
# - Rellenar con ceros a la izquierda si tiene menos de 8 dígitos

# Convertir a string:
df_filtrado['numero_documento'] = df_filtrado['numero_documento'].astype(str)

# Quitar espacios en blanco:
df_filtrado['numero_documento'] = df_filtrado['numero_documento'].str.strip()

# Rellenar con ceros a la izquierda (solo si tiene menos de 8 dígitos):
df_filtrado['numero_documento'] = df_filtrado['numero_documento'].str.zfill(8)

# Eliminar filas donde el DNI no tiene exactamente 8 dígitos o no es numérico
df_filtrado = df_filtrado[df_filtrado['numero_documento'].str.match(r'^\d{8}$')]

# 4. Guardar la base limpia (opcional)
df_filtrado.to_csv("basematricula.csv", index=False)


C:\Users\lisbeth\AppData\Local\Temp\ipykernel_30372\3165112083.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado['numero_documento'] = df_filtrado['numero_documento'].astype(str)
C:\Users\lisbeth\AppData\Local\Temp\ipykernel_30372\3165112083.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado['numero_documento'] = df_filtrado['numero_documento'].str.strip()
C:\Users\lisbeth\AppData\Local\Temp\ipykernel_30372\3165112083.py:20: SettingWithCopyWarning: 
A value is trying to be set o

Merge: Matriculados con postulantes 

In [2]:
# 1. Cargar las bases
df_matricula = pd.read_csv("basematricula.csv")

# Renombrar la columna
df_matricula = df_matricula.rename(columns={'numero_documento': 'DNI'})
print(df_matricula.columns)

Index(['codigo_modular', 'universidad', 'id_persona', 'tipo_documento', 'DNI',
       'edad', 'sexo', 'id_anio', 'abreviatura_anio', 'id_periodo_matricula',
       'periodo_matricula', 'id_local_carrera', 'codigo_ubigeo',
       'departamento', 'provincia', 'distrito', 'id_carrera', 'nombre_carrera',
       'codigo_carrera_inei', 'nombre_carrera_inei',
       'periodo_primera_matricula', 'periodo_ingreso',
       'domicilio_codigo_ubigeo', 'domicilio_departamento',
       'domicilio_provincia', 'domicilio_distrito'],
      dtype='object')


In [3]:
df_postulantes = pd.read_excel("1.Postulantes BC.xlsx")
print(df_postulantes.columns)

Index(['N°', 'DNI', 'APELLIDOS Y NOMBRES', 'REGIÓN DE ESTUDIO', 'IES', 'SEDE',
       'CARRERA', 'PUNTAJE FINAL', 'CONDICIÓN', 'RESTRICCION '],
      dtype='object')


In [4]:
# 2. Estandarizar DNI
df_postulantes['DNI'] = df_postulantes['DNI'].astype(str).str.strip().str.zfill(8)
df_matricula['DNI'] = df_matricula['DNI'].astype(str).str.strip().str.zfill(8)

# 3. Crear tabla pivote de matrícula
df_matricula['valor'] = 1
df_pivot = df_matricula.pivot_table(index='DNI', columns='periodo_matricula', values='valor', fill_value=0).reset_index()

  # Asegurar columnas necesarias existen
for col in ['2020-2', '2021-1', '2021-2']:
    if col not in df_pivot.columns:
        df_pivot[col] = 0
    

In [5]:
# 3. Crear tabla pivote de matrícula
df_matricula['valor'] = 1
df_pivot = df_matricula.pivot_table(index='DNI', columns='periodo_matricula', values='valor', fill_value=0).reset_index()

  # Asegurar columnas necesarias existen
for col in ['2020-2', '2021-1', '2021-2']:
    if col not in df_pivot.columns:
        df_pivot[col] = 0
    

In [6]:
# 4. Obtener variables adicionales por DNI desde la base de matrícula
columnas_extra = [
    'DNI', 'edad', 'sexo', 'codigo_ubigeo', 'departamento', 'provincia',
    'distrito', 'periodo_ingreso', 'domicilio_departamento', 'domicilio_distrito'
]

df_info_adicional = df_matricula[columnas_extra].drop_duplicates(subset='DNI')
        

In [7]:
# 5. Merge de info adicional + indicadores de matrícula
df_matricula_info = df_pivot.merge(df_info_adicional, on='DNI', how='left')

In [8]:
# 6. Merge con postulantes (mantener todos)
df_final = df_postulantes.merge(df_matricula_info, on='DNI', how='left')

In [9]:
# 7. Completar NaN en columnas de matrícula
for col in ['2020-2', '2021-1', '2021-2']:
    df_final[col] = df_final[col].fillna(0).astype(int)


In [10]:
# 8. Guardar resultado
df_final.to_csv("base_merge.csv", index=False)

In [11]:
# Verificar que tenemos todos los postulantes: 
print("Total de postulantes:", len(df_postulantes))
print("Total en el merge final:", len(df_final))


Total de postulantes: 7098
Total en el merge final: 7098


In [12]:
print(df_final.columns)

Index(['N°', 'DNI', 'APELLIDOS Y NOMBRES', 'REGIÓN DE ESTUDIO', 'IES', 'SEDE',
       'CARRERA', 'PUNTAJE FINAL', 'CONDICIÓN', 'RESTRICCION ', '2020-2',
       '2021-1', '2021-2', 'edad', 'sexo', 'codigo_ubigeo', 'departamento',
       'provincia', 'distrito', 'periodo_ingreso', 'domicilio_departamento',
       'domicilio_distrito'],
      dtype='object')


In [13]:
# 1. Cargar base para eliminar 
df_delete = pd.read_excel("2.delete.xlsx")

In [14]:
# 2. Estandarizar DNIs
df_final['DNI'] = df_final['DNI'].astype(str).str.strip().str.zfill(8)
df_delete['DNI'] = df_delete['DNI'].astype(str).str.strip().str.zfill(8)

In [15]:
# 3. Eliminar todas las filas cuyo DNI esté en la base 'delete'
df_sin_duplicados = df_final[~df_final['DNI'].isin(df_delete['DNI'])]

In [16]:
# 4. Guardar nueva base
df_sin_duplicados.to_csv("Base_limpia.csv", index=False)
df_final = df_sin_duplicados

In [17]:
# 5. Mostrar resumen
print(f"Filas antes de eliminar duplicados: {len(df_final)}")
print(f"Filas eliminadas por estar en 'delete': {len(df_final) - len(df_sin_duplicados)}")
print(f"Filas finales: {len(df_sin_duplicados)}")

Filas antes de eliminar duplicados: 7039
Filas eliminadas por estar en 'delete': 0
Filas finales: 7039


In [18]:
print(df_final.columns)

Index(['N°', 'DNI', 'APELLIDOS Y NOMBRES', 'REGIÓN DE ESTUDIO', 'IES', 'SEDE',
       'CARRERA', 'PUNTAJE FINAL', 'CONDICIÓN', 'RESTRICCION ', '2020-2',
       '2021-1', '2021-2', 'edad', 'sexo', 'codigo_ubigeo', 'departamento',
       'provincia', 'distrito', 'periodo_ingreso', 'domicilio_departamento',
       'domicilio_distrito'],
      dtype='object')


Juntamos la base con una base de calidad por universidad

In [19]:
# 1. Cargar archivo de calidad universitaria
df_calidad = pd.read_excel("3.calidad universitaria.xlsx")

In [20]:
# 2. Estandarizar columnas para unir (opcional: quitar espacios, pasar a mayúsculas, etc.)
df_final['IES'] = df_final['IES'].astype(str).str.strip()
df_final['carrera'] = df_final['CARRERA'].astype(str).str.strip()
df_calidad['IES'] = df_calidad['IES'].astype(str).str.strip()
df_calidad['carrera'] = df_calidad['CARRERA'].astype(str).str.strip()


In [21]:
print(df_calidad.columns)

Index(['IES', 'CARRERA', 'TOP ', 'Familia', 'carrera'], dtype='object')


In [22]:
# 3. Crear los dos DataFrames de referencia
df_top = df_calidad[['IES', 'TOP ']].drop_duplicates()
df_familia = df_calidad[['CARRERA', 'Familia']].drop_duplicates()

In [23]:
# 4. Merge con la base final
df_final = df_final.merge(df_top, on='IES', how='left')
df_final = df_final.merge(df_familia, on='CARRERA', how='left')

In [24]:
# 5. Guardar nueva base
df_final.to_csv("base.csv", index=False)

print("Columnas 'TOP' y 'Familia' agregadas exitosamente.")


Columnas 'TOP' y 'Familia' agregadas exitosamente.


In [25]:
print(df_final.columns)

Index(['N°', 'DNI', 'APELLIDOS Y NOMBRES', 'REGIÓN DE ESTUDIO', 'IES', 'SEDE',
       'CARRERA', 'PUNTAJE FINAL', 'CONDICIÓN', 'RESTRICCION ', '2020-2',
       '2021-1', '2021-2', 'edad', 'sexo', 'codigo_ubigeo', 'departamento',
       'provincia', 'distrito', 'periodo_ingreso', 'domicilio_departamento',
       'domicilio_distrito', 'carrera', 'TOP ', 'Familia'],
      dtype='object')


In [26]:
print(df.tail()) 

         codigo_modular                                   universidad  \
6989703       260000059                     UNIVERSIDAD ALAS PERUANAS   
6989704       260000047                        UNIVERSIDAD DE HUANUCO   
6989705       160000051          UNIVERSIDAD NACIONAL DE HUANCAVELICA   
6989706       160000001      UNIVERSIDAD NACIONAL MAYOR DE SAN MARCOS   
6989707       260000053  UNIVERSIDAD CATÓLICA LOS ÁNGELES DE CHIMBOTE   

         id_persona tipo_documento numero_documento  edad       sexo  id_anio  \
6989703     4087691            DNI         74935088    19   FEMENINO       50   
6989704     3557193            DNI         48512172    29  MASCULINO       50   
6989705     1720463            DNI         74077729    27   FEMENINO       57   
6989706     3491869            DNI         72127226    20  MASCULINO       57   
6989707      623958            DNI         40367987    42  MASCULINO       57   

         abreviatura_anio  id_periodo_matricula  ... id_carrera  \
6989703

In [27]:
print(df.head(10)) 

   codigo_modular                                   universidad  id_persona  \
0       260000055                 UNIVERSIDAD PRIVADA DEL NORTE     5034082   
1       260000055                 UNIVERSIDAD PRIVADA DEL NORTE     5033730   
2       260000053  UNIVERSIDAD CATÓLICA LOS ÁNGELES DE CHIMBOTE     3090708   
3       160000012            UNIVERSIDAD NACIONAL DEL ALTIPLANO     1145374   
4       260000038                     UNIVERSIDAD PERUANA UNIÓN      823961   
5       160000023        UNIVERSIDAD NACIONAL HERMILIO VALDIZAN       33014   
6       260000057             UNIVERSIDAD SAN IGNACIO DE LOYOLA     2038190   
7       260000046            UNIVERSIDAD PRIVADA ANTENOR ORREGO      919160   
8       260000037                 UNIVERSIDAD PERUANA LOS ANDES      397312   
9       260000059                     UNIVERSIDAD ALAS PERUANAS     4349560   

  tipo_documento numero_documento  edad       sexo  id_anio  abreviatura_anio  \
0            DNI         71756035    18  MASCULIN

In [28]:
print(df_final['TOP '].unique())

['NO TOP 30' 'TOP 30']


In [29]:
print(df_final['TOP '].isna().sum())       # Cantidad de NaN en TOP
print(df_final['Familia'].isna().sum())   # Cantidad de NaN en Familia

0
0


In [30]:
df_final.to_csv("../data/base.csv", index=False)


In [31]:
df_final

,N°,DNI,APELLIDOS Y NOMBRES,REGIÓN DE ESTUDIO,IES,SEDE,CARRERA,PUNTAJE FINAL,CONDICIÓN,RESTRICCION,...,codigo_ubigeo,departamento,provincia,distrito,periodo_ingreso,domicilio_departamento,domicilio_distrito,carrera,TOP,Familia
0,2,70801736,"ABAD GARCIA, MARISOL",LIMA\nMETROPOLITANA,UNIVERSIDAD PRIVADA DEL NORTE,SEDE COMAS,ADMINISTRACION Y NEGOCIOS INTERNACIONALES,55,NO SELECCIONADO,LIMA,...,150110.0,LIMA,LIMA,COMAS,2017-1,LIMA,CARABAYLLO,ADMINISTRACION Y NEGOCIOS INTERNACIONALES,NO TOP 30,"Ciencias Naturales, Exactas y de la Computación"
1,3,71469634,"ABAD MARINA, ROCIO DEL PILAR",LIMA\nMETROPOLITANA,UNIVERSIDAD ANTONIO RUIZ DE MONTOYA,SEDE PUEBLO LIBRE,DERECHO (4),60,NO SELECCIONADO,LIMA,...,150121.0,LIMA,LIMA,PUEBLO LIBRE,2018-1,SAN MARTIN,MOYOBAMBA,DERECHO (4),TOP 30,"Ciencias Sociales, Comerciales y Derecho"
2,6,60736859,"ABAD VASQUEZ, DAYANA ANTUANET",LIMA\nMETROPOLITANA,UNIVERSIDAD PRIVADA DEL NORTE,SEDE LOS OLIVOS,ADMINISTRACION,50,NO SELECCIONADO,LIMA,...,150117.0,LIMA,LIMA,LOS OLIVOS,2017-2,LIMA,CARABAYLLO,ADMINISTRACION,NO TOP 30,"Ciencias Sociales, Comerciales y Derecho"
3,7,71332025,"ABAD VELIZ, JOSE FRANCISCO",LIMA\nMETROPOLITANA,UNIVERSIDAD TECNOLOGICA DEL PERU - UTP,SEDE VILLA EL SALVADOR,DERECHO (4),50,NO SELECCIONADO,LIMA,...,150142.0,LIMA,LIMA,VILLA EL SALVADOR,2018-1,LIMA,VILLA EL SALVADOR,DERECHO (4),NO TOP 30,"Ciencias Sociales, Comerciales y Derecho"
4,8,74855443,"ABAD CHATE, JIREH ANGELICA ARELY",LIMA\nMETROPOLITANA,UNIVERSIDAD PRIVADA SAN IGNACIO DE LOYOLA,SEDE LA MOLINA,ARTE Y DISEÃ‘O EMPRESARIAL,70,SELECCIONADO,LIMA,...,150114.0,LIMA,LIMA,LA MOLINA,2016-1,LIMA,LURIGANCHO,ARTE Y DISEÃ‘O EMPRESARIAL,NO TOP 30,Humanidades y Artes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7045,13097,75825475,"AGUILAR CERNA, NINFA GRACE\n",LIMA\nMETROPOLITANA,UNIVERSIDAD TECNOLOGICA DEL PERU - UTP\nNABEC,SEDE LOS OLIVOS\n,PSICOLOGIA\n,50,NO SELECCIONADO,LIMA,...,150117.0,LIMA,LIMA,LOS OLIVOS,2017-2,LIMA,COMAS,PSICOLOGIA,NO TOP 30,"Ciencias Sociales, Comerciales y Derecho"
7046,13098,76825449,"CABOS LINARES, DIANA LIZETH\n",LIMA\nMETROPOLITANA,UNIVERSIDAD CESAR VALLEJO,SEDE ATE\n,DERECHO (4),50,NO SELECCIONADO,LIMA,...,150103.0,LIMA,LIMA,ATE,2019-2,CAJAMARCA,SAN BERNARDINO,DERECHO (4),NO TOP 30,"Ciencias Sociales, Comerciales y Derecho"
7047,13099,40288773,"CHUMBIRIZA GONZALES, CARLOS\n",LIMA\nMETROPOLITANA,UNIVERSIDAD AUTONOMA DEL PERU,SEDE VILLA EL SALVADOR\n,PSICOLOGIA\n,55,NO SELECCIONADO,LIMA,...,150142.0,LIMA,LIMA,VILLA EL SALVADOR,2016-2,LIMA,VILLA EL SALVADOR,PSICOLOGIA,NO TOP 30,"Ciencias Sociales, Comerciales y Derecho"
7048,13100,75886632,"GONZALES GARCIA, ANGIE KARLA\n",LIMA\nMETROPOLITANA,UNIVERSIDAD ANTONIO RUIZ DE MONTOYA,SEDE PUEBLO LIBRE\n,DERECHO (4),60,NO SELECCIONADO,LIMA,...,150121.0,LIMA,LIMA,PUEBLO LIBRE,2017-2,LIMA,SAN MARTIN DE PORRES,DERECHO (4),TOP 30,"Ciencias Sociales, Comerciales y Derecho"


In [32]:
df_final['CONDICIÓN'].value_counts()


SELECCIONADO       5052
NO SELECCIONADO    1998
Name: CONDICIÓN, dtype: int64

In [33]:
df.shape

(6989708, 26)